In [1]:
# ── Percorsi ─────────────────────────────────────────────────────────────
JSON_PATH  = r"C:\\Users\\angel\\OneDrive\\Desktop\\ProgettoNLP\\progetto\\dataset\\VQA_RAD Dataset Public.json"
IMAGES_DIR = r"C:\\Users\\angel\\OneDrive\\Desktop\\ProgettoNLP\\progetto\\dataset\\VQA_RAD Image Folder"
OUTPUT_DIR = r"C:\\Users\\angel\\OneDrive\\Desktop\\ProgettoNLP\\progetto\\dataset\\VQA_RAD Output Folder"

In [ ]:
import json
from pathlib import Path
from PIL import Image
from datasets import Dataset, DatasetDict, Features, Value, Image as HFImage


# ── Carica JSON ───────────────────────────────────────────────────────────
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Record nel JSON: {len(data)}")

# ── Costruisci le righe ───────────────────────────────────────────────────
images_path = Path(IMAGES_DIR)
rows = []
missing = []

for rec in data:
    img_name = rec.get("image_name")
    img_path = images_path / img_name if img_name else None

    if img_path and img_path.exists():
        pil_img = Image.open(img_path).convert("RGB")
    else:
        missing.append(img_name)
        pil_img = None

    rows.append({
        "qid":            str(rec.get("qid", "")),
        "image_name":     img_name,
        "image":          pil_img,
        "image_organ":    rec.get("image_organ"),
        "question":       rec.get("question"),
        "question_type":  rec.get("question_type"),
        "phrase_type":    rec.get("phrase_type"),
        "answer":         rec.get("answer"),
        "answer_type":    rec.get("answer_type"),
    })

print(f"Immagini mancanti: {len(missing)}")

# ── Crea Dataset ──────────────────────────────────────────────────────────
cols = {key: [row[key] for row in rows] for key in rows[0].keys()}

features = Features({
    "qid":           Value("string"),
    "image_name":    Value("string"),
    "image":         HFImage(),
    "image_organ":   Value("string"),
    "question":      Value("string"),
    "question_type": Value("string"),
    "phrase_type":   Value("string"),
    "answer":        Value("string"),
    "answer_type":   Value("string"),
})

ds = Dataset.from_dict(cols, features=features)
print(f"\nDataset creato: {ds}")

# ── Split train / validation / test (70 / 15 / 15) ───────────────────────
split_tv = ds.train_test_split(test_size=0.30, seed=42)
split_vt = split_tv["test"].train_test_split(test_size=0.50, seed=42)

dataset_dict = DatasetDict({
    "train":      split_tv["train"],
    "validation": split_vt["train"],
    "test":       split_vt["test"],
})

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
dataset_dict.save_to_disk(OUTPUT_DIR)
print(f"\nDataset salvato in: {OUTPUT_DIR}")
print(f"  train:      {len(dataset_dict['train'])} esempi")
print(f"  validation: {len(dataset_dict['validation'])} esempi")
print(f"  test:       {len(dataset_dict['test'])} esempi")

Record nel JSON: 2248
Immagini mancanti: 0

Dataset creato: Dataset({
    features: ['qid', 'image_name', 'image', 'image_organ', 'question', 'question_type', 'phrase_type', 'answer', 'answer_type'],
    num_rows: 2248
})


Saving the dataset (1/1 shards): 100%|██████████| 338/338 [00:00<00:00, 821.11 examples/s]


Dataset salvato in: C:\Users\angel\OneDrive\Desktop\ProgettoNLP\progetto\dataset\VQA_RAD Output FolderValidation
  train:      1573 esempi
  validation: 337 esempi
  test:       338 esempi


In [5]:
from huggingface_hub import login, upload_folder

login()
upload_folder(folder_path=OUTPUT_DIR, repo_id="Angelo0102/VQA-RAD", repo_type="dataset")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/Angelo0102/VQA-RAD/commit/6547d9bd30d3d0bd0807ce6b9e3702accd749478', commit_message='Upload folder using huggingface_hub', commit_description='', oid='6547d9bd30d3d0bd0807ce6b9e3702accd749478', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Angelo0102/VQA-RAD', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Angelo0102/VQA-RAD'), pr_revision=None, pr_num=None)

In [6]:
from datasets import load_from_disk
ds = load_from_disk(OUTPUT_DIR)

sample = ds["train"][0]
print(type(sample["image"]))   
sample["image"].show()          
print(sample["question"])
print(sample["answer"])

<class 'PIL.PngImagePlugin.PngImageFile'>
Is there a fracture?
No
